### 问题描述：
######  生成的数据形式（含有变系数）

$$ Y =\beta_0+ \beta_1 * X_1+ \beta_2  * X_2 + \cdots + \beta_{10}  * X_{10} + \alpha_1 (u) * Z_1+ \alpha_2(u) * Z_2 + \dots  + \alpha_7(u) * Z_7 + \epsilon $$

其中：
$
\beta_0 = 1\\
\beta_1 = 1\\
\beta_2 = 1\\
\beta_3 = \cdots = \beta_{10} = 0 \\
\alpha_1 (x_1) =
\begin{cases}
1,&0.5> x_1 \geq0 \\ 
-1,&1 > x_1 \geq0.5
\end{cases}\\
\alpha_2 (x_1) =
\begin{cases}
-1.5,&0.5> x_1 \geq0 \\ 
0,&1 > x_1 \geq0.5 
\end{cases}\\
\cdots\\
\alpha_7 (x_1) =
\begin{cases}
-1.5,&0.5> x_1 \geq0 \\ 
0,&1 > x_1 \geq0.5 
\end{cases}\\
$

$X_1  服从  U(0,1)$

$X_2  服从  N(0,1)$

$\cdots$

$X_{10}  服从  N(0,1)$

$Z_1  服从  B(0.5)$

$Z_2  服从  B(0.6)$

\cdots

$Z_7  服从  B(0.6)$

$\epsilon  服从  N(0,0.5)$

##### 简单线性模型，变系数判别为分段函数，beta1与beta2的亚组数一致(组数为2)，变点条件一致  
####  n = 400; replication = 200 ;KNN ~ K = 5

##### p = o(n),选择p = n^{1/3} = 7 q = 10

In [1]:
#导入相关库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import seaborn as sns
n = 400 #训练样本
n1 = 200#预测样本
from pandas import read_csv
import math
import random
import csv

import numpy as np
import pandas as pd
from numpy import matrix

from sklearn.svm import SVC
import statsmodels.api as sm
import pylab as pl
import scipy.linalg as sl

def I_12(x):
    if x >= 0:
        return 1
    else:
        return 0

def I_11(x):
    if x < 0:
        return 1
    else:
        return 0

def I_22(x):
    if x >= 0:
        return 1
    else:
        return 0


def I_21(x):
    if x < 0:
        return 1
    else:
        return 0

def I_12or(x):
    if x >= 0:
        return 1
    else:
        return 0

def I_11or(x):
    if x < 0:
        return 1
    else:
        return 0

def I_22or(x):
    if x >= -1:
        return 1
    else:
        return 0


def I_21or(x):
    if x< -1:
        return 1
    else:
        return 0
    
# Soft Thresholding
def ST(b, lam):
    st = np.sign(b) * max(abs(b) - lam, 0)
    return st
def MCP(a,b,gamma,rho,lam):
    if abs(a)<= gamma*rho*b:
        return ST(a,b)/(1-1/(gamma*lam/b))
    else:
        return a
from scipy.special import comb
def rand_index_score(clusters, classes):
    tp_plus_fp = comb(np.bincount(clusters), 2).sum()
    tp_plus_fn = comb(np.bincount(classes), 2).sum()
    A = np.c_[(clusters, classes)]
    tp = sum(comb(np.bincount(A[A[:, 0] == i, 1]), 2).sum()
            for i in set(clusters))
    fp = tp_plus_fp - tp
    fn = tp_plus_fn - tp
    tn = comb(len(A), 2) - tp - fp - fn
    return (tp + tn) / (tp + fp + fn + tn)

def mcp_threshold(S_g, H_g, lam, gamma_beta):
    """MCP阈值规则更新系数"""
    abs_Sg = np.abs(S_g)
    sign_Sg = np.sign(S_g) if S_g != 0 else 1
    if abs_Sg <= lam * H_g:
        return 0.0
    elif abs_Sg > gamma_beta * lam * H_g:
        return S_g / H_g
    else:
        return (S_g - sign_Sg * lam) / (H_g - 1/gamma_beta)
    
from scipy.linalg import block_diag

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [2]:
#重复循环50次试一下
np.random.seed(1809)
import time
start = time.perf_counter()
m = 200
beta_chucun = np.zeros((m,3))
betaor_chucun = np.zeros((m,3))
for i in range(1, 8):    # i 控制第一个数字 (1~2)
    for j in range(1, 3): # j 控制第二个数字 (1~3)
        globals()[f'alpha{i}_{j}cunchu'] = []
        globals()[f'alphaor{i}_{j}cunchu'] = []
ACC_finalcunchu = []
ACC_prefinalcunchu = []

for i in range(1,8):
    globals()[f'ACC{i}_cunchu'] = np.zeros((m,1))
    globals()[f'ACCp{i}_cunchu'] = np.zeros((m,1))
    globals()[f'Pre{i}_cunchu'] = np.zeros((m,10000))
RI_chucun = np.zeros((m,7))
K_chucun = np.zeros((m,7))
for t in range(200):
    print(t) 
    # 设置样本量
    n = 400  # 可根据需要调整
    # 生成 X 变量
    X0 = np.ones(n)
    X1 = np.random.uniform(0, 1, n)
    X2 = np.random.normal(0, 1, n)
    X3_X10 = np.random.normal(0, 1, (n, 8))  # X3到X10

    random_u =  X1

    Z1 = np.random.normal(loc=1, scale=1, size=n)

    # 生成 Z2 到 Z7（正态分布，均值=1，方差=1，共6列）
    Z2_Z7 = np.random.normal(loc=1, scale=1, size=(n, 6))

    # 计算 alpha 系数
    alpha1 = np.where(X1 < 0.5, 1, -1)
    alpha2_to_7 = np.where(X1 < 0.5, -2, 0)

    # 计算交互项
    interaction_terms = alpha1 * Z1
    for i in range(6):  # 对应 Z2到Z7
        interaction_terms += alpha2_to_7 * Z2_Z7[:, i]

    # 计算 Y
    epsilon = np.random.normal(0, np.sqrt(0.5), n)
    Y = 1 + X1 + X2 + interaction_terms + epsilon

    # 构建 DataFrame
    index = []
    for i in range(n):
        index_1 = i
        index.append(int(index_1)) 
    data = pd.DataFrame({'index':index,'Y': Y,'X0': X0, 'X1': X1, 'X2': X2})
    for i in range(3, 11):
        data[f'X{i}'] = X3_X10[:, i-3]
    for i in range(1, 8):
        if i == 1:
            data[f'Z{i}'] = Z1
        else:
            data[f'Z{i}'] = Z2_Z7[:, i-2]
    data[f'U'] = random_u
    for i  in range(1,8):
        if i == 1:
            data[f'alpha{i}'] = alpha1
        else:
            data[f'alpha{i}'] = alpha2_to_7

    data.to_csv('p high dimension2.csv',index=False,header=True)
    #print(data.head())
    #data

    #生成预测数据集
    n1 = 200  # 可根据需要调整

    # 生成 X 变量
    X0 = np.ones(n1)
    X1 = np.random.uniform(0, 1, n1)
    X2 = np.random.normal(0, 1, n1)
    X3_X10 = np.random.normal(0, 1, (n1, 8))  # X3到X10

    random_u =  X1

    # 生成 Z 变量
    Z1 = np.random.binomial(1, 0.5, n1).astype(float)  # 转为浮点
    Z2_Z7 = np.random.binomial(1, 0.6, (n1, 6)).astype(float)  # 转为浮点

    # 计算 alpha 系数
    alpha1 = np.where(X1 < 0.5, 1, -1)
    alpha2_to_7 = np.where(X1 < 0.5, -2, 0)

    # 计算交互项
    interaction_terms = alpha1 * Z1
    for i in range(6):  # 对应 Z2到Z7
        interaction_terms += alpha2_to_7 * Z2_Z7[:, i]

    # 计算 Y
    epsilon = np.random.normal(0, np.sqrt(0.5), n1)
    Y = 1 + X1 + X2 + interaction_terms + epsilon

    index = []
    for i in range(n1):
        index_1 = i
        index.append(int(index_1))

    # 构建 datapreFrame
    datapre = pd.DataFrame({'index': index,'Y': Y, 'X0': X0, 'X1': X1, 'X2': X2})
    for i in range(3, 11):
        datapre[f'X{i}'] = X3_X10[:, i-3]
    for i in range(1, 8):
        if i == 1:
            datapre[f'Z{i}'] = Z1
        else:
            datapre[f'Z{i}'] = Z2_Z7[:, i-2]
    datapre[f'U'] = random_u
    for i  in range(1,8):
        if i == 1:
            datapre[f'alpha{i}'] = alpha1
        else:
            datapre[f'alpha{i}'] = alpha2_to_7

    datapre.to_csv('p high dimension pre2.csv',index=False,header=True)
    
    filenamepre = 'p high dimension pre2.csv'
    datapre = read_csv(filenamepre)
    #生成判罚阵

    # 导入随机生成的数据
    filename = 'p high dimension2.csv'
    data = read_csv(filename)

    with open('p high dimension2.csv','r')as file:#'r'为读取文本
        reader = csv.DictReader(file)
        datas = [row for row in reader]

    def distance(d1, d2):
        """2范数，即欧几里得距离
           1范数，曼哈顿距离"""
        res = 0
        for key in("U"):
            res+=abs((float(d1[key])-float(d2[key])))#一范数
            #res+=(float(d1[key])-float(d2[key]))**2  #二范数
        return res
    #定义距离函数
    #影响knn的一般为距离的定义和k的取值，所以我把距离函数单独定义一下，这样方便更换

    #KNN计算离点最近的K-1个点，这是因为此时我在计算距离时并没有把自身排除，在排序时总有它自身距离自身为0的一个点
    K = 5  #以2-NN为例
    def KNN(data):
        #1.距离
        res = [
            {"index": train['index'],"distance": distance(data,train)}
            #diatance(data,train)
            for train in datas
        ]
        #2升序排列
        res = sorted(res , key = lambda item:item["distance"])
         #3.取前K个
        res2 = res[0:K]
        r2 = []
        for r in res2:
            r_1 = r["index"]
            r2.append(r_1)
        #print(res2)
        #print(r2)
        return r2
    k = 4#与前面的K-1相等
    G = np.zeros(shape=(n*k,n))#nk*n维矩阵
    #G  生成nk*n的零矩阵

    for i in range(0,n):
        for m in range(0,k):
            G[m+k*i][i] =  1
            ##k行一个元素为1 
    #G

    for j in range(0,n):
        list1 = KNN(datas[j])
        list2 = list1[1:]#取最近的K-1 = k个值
        for i in range(0,k):
            r_12 = int(float(list2[i]))
            G[i+j*k][r_12] = -1
            #G[i+j][#knn的那几个元素] = -1 
    #生成所需要的G矩阵
    
    #给初值

    lam = 0.0001
    m = 7
    I_p = np.diag(np.ones(m))
    A = np.kron(G,I_p)

    # 提取 X1 到 X10 的数据
    X = data[[f'X{i}' for i in range(0, 11)]].values  # 形状 (n_samples, 11)
    #X = data[['X1', 'X2', 'X3']].values  # 原代码中X0列不存在，改为X1,X2,X3
    Z = data[[f'Z{i}' for i in range(1,8)]].values
    Y_1 = data['Y'].values  # 原代码列名应为Y而非Y1

    I_n = np.diag(np.ones(n))
    Q_1 = np.dot(X,np.linalg.pinv(np.dot(X.T,X)))
    Q_X = I_n - np.dot(Q_1,X.T)

    Z00 = []
    for i in range(n):
        Z00 = sl.block_diag(Z00,Z[i])
        i = i+1
    Z00 = np.delete(Z00, 0, 0) #块对角化.Z = (Z_1^T,...,Z_n^T)

    alpha_intial = np.dot(np.linalg.pinv(np.dot(np.dot(Z00.T,Q_X),Z00)+ lam * np.dot(A.T,A)),np.dot(np.dot(Z00.T,Q_X),Y_1))

    beta_intial = np.dot(np.dot(np.linalg.pinv(np.dot(X.T,X)),X.T),Y_1 - np.dot(Z00,alpha_intial))

    # 解包 alpha 系数
    alpha_intial_2d = alpha_intial.reshape(n, m)  # 形状 (n, 7)
    alpha11, alpha22, alpha33, alpha44, alpha55, alpha66, alpha77 = alpha_intial_2d.T
    # 提取真实alpha值 (需确认数据中存在这些列)
    alpha_names = [f'alpha{i}' for i in range(1,8)]
    alpha_true = [data[col].values for col in alpha_names]

    # 创建字典映射：键为Z1-Z7，值为对角矩阵
    Z_matrices = {f'Z{i}': np.diag(data[f'Z{i}'].values) for i in range(1, 8)}
    # 访问示例：
    Z_1 = Z_matrices['Z1']
    Z_2 = Z_matrices['Z2']

    Z_3 = Z_matrices['Z3']
    Z_4 = Z_matrices['Z4']

    Z_5 = Z_matrices['Z5']
    Z_6 = Z_matrices['Z6']
    Z_7 = Z_matrices['Z7']
    # 将Z_1到Z_7存储到列表中（假设已预先定义）
    Z_list = [Z_1, Z_2, Z_3, Z_4, Z_5, Z_6, Z_7]
    # 创建alpha集合
    alphas = np.array([alpha11, alpha22, alpha33, alpha44, alpha55, alpha66, alpha77])
    #alpha = alphas.T
    epsilon = 1e-3
    k=0
    lambda_s2 = 0.2  # 惩罚系数（需通过交叉验证调整）
    gamma_beta = 2 

    # 计算delta集合
    deltas = [np.dot(G, alpha) for alpha in alphas]
    delta = np.array(deltas).T
    # 初始化mu集合 (等效于全零数组)
    n_samples = len(data)
    mu = [np.zeros(4*n_samples) for _ in range(7)]
    mu = np.array(mu).T
    rho = 2
    lam = 1
    gamma = 2
    alpha = alphas.T
    beta = beta_intial
    #每步计算结果
    while k < 50:#迭代
        n_samples, n_features = X.shape
        #beta = np.zeros(n_features)  # 初始化为零向量
        H = np.sum(X**2, axis=0)     # 预计算每列的平方和

        Y_prime =  Y_1 - (Z * alpha).sum(axis=1)
        for g in range(n_features):
            if H[g] == 0:  # 避免除零错误
                continue
            # 计算排除当前系数的残差
            r = Y_prime - X @ beta + X[:, g] * beta[g]
            S_g = X[:, g].T @ r
            # 应用MCP阈值规则更新系数
            beta[g] = mcp_threshold(S_g, H[g], lambda_s2, gamma_beta)

        #beta = coordinate_descent_mcp(X, Y_prime, lambda_s2, gamma_beta)

        # 将结果解包到原变量（如果后续代码需要单独变量名）
        #delta1, delta2, delta3, delta4, delta5, delta6, delta7 = deltas
        #mu1, mu2, mu3, mu4, mu5, mu6, mu7 = mu

        Y_a  = Y_1 - X @ beta
        # 更新所有alpha分量（共7个组）
        for g in range(7):  # g从0到6，对应组1到7
            # 1. 计算排除当前组的残差
            residual = Y_a.copy()
            for j in range(7):
                if j != g:
                    residual -= Z_list[j] @ alpha[:, j]  # 减去其他组的贡献

            # 2. 计算岭回归项和ADMM项
            Z_g = Z_list[g]
            GTG = G.T @ G  # 预计算公共项（若G不变可提前计算）
            inv_matrix = np.linalg.pinv(Z_g.T @ Z_g + rho * GTG)

            # ADMM参数提取
            delta_g = delta[:, g]
            mu_g = mu[:, g]
            admm_term = rho * G.T @ (delta_g - (1/rho) * mu_g)

            # 3. 更新alpha[:,g]
            alpha[:, g] = inv_matrix @ (Z_g.T @ residual + admm_term)     

        for i in range(7):
            bb = np.dot(G, alpha[:,i]) + 1/rho * mu[:,i]
            ras = [MCP(b, lam/rho, gamma, rho, lam) for b in bb[:4*n]]  # 假设4n不超过bb的长度
            delta[:,i] = np.asarray(ras)
        # 批量更新mu
        for i in range(7):
            mu[:,i] = mu[:,i] + rho * (np.dot(G, alpha[:,i]) - delta[:,i])      
        #停止迭代的条件 
        k = k+1
        #print(k)
        if np.linalg.norm(mu[:,5])/(4*n) < epsilon and np.linalg.norm(mu[:,6])/(4*n) < epsilon:
            break
        else:
            alpha_old = alpha
            #迭代过程
            
    #F1.append(beta)#
    # 定义参数配置列表：每个alpha对应的差分阈值和百分比阈值
    alpha_configs = [
        {'diff_thresh': 0.5, 'percent_thresh': 0.1},  # alpha1
        {'diff_thresh': 0.4, 'percent_thresh': 0.05},  # alpha2
        {'diff_thresh': 0.4, 'percent_thresh': 0.05},  # alpha3
        {'diff_thresh': 0.4, 'percent_thresh': 0.1},   # alpha4
        {'diff_thresh': 0.4, 'percent_thresh': 0.1},   # alpha5
        {'diff_thresh': 0.5, 'percent_thresh': 0.05},  # alpha6
        {'diff_thresh': 0.4, 'percent_thresh': 0.1}     # alpha7
    ]

    # 将alpha1-alpha7存入列表（假设已预先定义）
    alphas = [alpha[:,0],alpha[:,1], alpha[:,2], alpha[:,3], alpha[:,4], alpha[:,5], alpha[:,6]]

    K_results = []
    # 统一处理所有alpha分量
    for idx, (alpha_f, config) in enumerate(zip(alphas, alpha_configs), 1):
        # 1. 排序并计算差分
        alpha_sort = np.sort(alpha_f)
        biandian = np.diff(alpha_sort)

        # 2. 定位跳变点
        location = np.array(np.where(abs(biandian) > config['diff_thresh']))
        location = np.insert(location, 0, 0)        # 头部插入0
        location = np.insert(location, len(location), 398)  # 尾部插入398

        # 3. 计算组数K         

        k_val = np.count_nonzero(np.diff(location) > config['percent_thresh'] * n)
        K_results.append(k_val)

    # 转换为NumPy数组并赋值
    K_results = np.array(K_results)
    K_chucun[t, :] = K_results #存储所有的K
    #print(K_results)
    
    # 读取数据
    filename = 'p high dimension2.csv'
    data = pd.read_csv(filename)

    # 初始化数据收集列表
    arrays = []

    # 提取基础列并重命名Y1为Y
    base_cols = ['index', 'Y'] + [f'Z{i}' for i in range(1,8)] + [f'X{i}' for i in range(0,11)]
    arrays += [data[col].values for col in base_cols]

    # 添加随机变量U
    arrays.append(data['U'])

    alpha1_before, alpha2_before, alpha3_before, alpha4_before,alpha5_before, alpha6_before, alpha7_before = (
        data[f'alpha{i}'].values for i in range(1,8)
    )
    # 添加原始alpha系数（alpha1_before到alpha7_before）

    alpha_before_vars = [alpha1_before, alpha2_before, alpha3_before,
                        alpha4_before, alpha5_before, alpha6_before, alpha7_before]
    arrays += alpha_before_vars

    #添加迭代后的alpha
    alpha_vars = [alpha[:,i].flatten() for i in range(7)]

    #alpha_vars = [np.squeeze(np.array(alpha_P)).flatten() for alpha_P in alpha_vars]
    arrays += alpha_vars
    #np.shape(np.array(arrays))
    #np.shape(arrays)
    #arrays.reshape(400,34)
    # 构建列名
    columns = (['index', 'Y'] +                     # 基础列
              [f'Z{i}' for i in range(1,8)] +       # Z1-Z7
              [f'X{i}' for i in range(0,11)] +  ['U']  +        # X系列和U
              [f'alpha{i}_origin' for i in range(1,8)] +  # 原始alpha
              [f'alpha{i}' for i in range(1,8)])     # 处理后的alpha

    # 创建DataFrame
    result = pd.DataFrame(np.array(arrays).T, columns=columns)
    # 使用安全方式构建
    #result = pd.DataFrame({col: arr for col, arr in zip(columns, arrays)})


    # RI计算配置列表（每个alpha的差异阈值和标签条件）
    ri_configs = [
        {'col': 'alpha1', 'diff_thresh': 0.4, 'true_cond': lambda x: 1 if x == -1 else 0},
        {'col': 'alpha2', 'diff_thresh': 0.4, 'true_cond': lambda x: 1 if x == 0 else 0},
        {'col': 'alpha3', 'diff_thresh': 0.4, 'true_cond': lambda x: 1 if x == 0 else 0},
        {'col': 'alpha4', 'diff_thresh': 0.4, 'true_cond': lambda x: 1 if x == 0 else 0},
        {'col': 'alpha5', 'diff_thresh': 0.4, 'true_cond': lambda x: 1 if x == 0 else 0},
        {'col': 'alpha6', 'diff_thresh': 0.4, 'true_cond': lambda x: 1 if x == 0 else 0},
        {'col': 'alpha7', 'diff_thresh': 0.4, 'true_cond': lambda x: 1 if x == 0 else 0}
    ]

    RI_results = []

    for idx, config in enumerate(ri_configs, 1):
        alpha_col = config['col']

        # 1. 按当前alpha排序
        result_sorted = result.sort_values(by=alpha_col, ascending=True).copy()

        # 2. 计算差分列
        diff_col = f'new_label{idx}diff'
        result_sorted[diff_col] = abs(result_sorted[alpha_col].diff(-1).shift())
        result_sorted = result_sorted.fillna(0).reset_index(drop=True)

        # 3. 定位变点
        dfsuoyin = result_sorted.loc[result_sorted[diff_col] > config['diff_thresh']]
        biandianxu = dfsuoyin.index.values
        biandianxu = np.insert(biandianxu, 0, 0)
        biandianxu = np.insert(biandianxu, len(biandianxu), 400)

        # 4. 生成预测标签
        labels_pred = np.zeros(400, int)
        for i in range(len(biandianxu)-1):
            labels_pred[biandianxu[i]:biandianxu[i+1]] = i
        labels_pred[biandianxu[-1]:] = len(biandianxu)

        # 5. 生成真实标签
        true_col = f'labels{idx}_true'
        origin_col = f'{alpha_col}_origin'
        result_sorted[true_col] = result_sorted[origin_col].apply(config['true_cond'])

        # 6. 计算RI
        RI_1 = rand_index_score(result_sorted[true_col].values, labels_pred)
        RI_results.append(RI_1)

    # 解包结果（可选）
    #RI1_1, RI2_1, RI3_1, RI4_1, RI5_1, RI6_1, RI7_1 = RI_results
    RI_results = np.array(RI_results)
    RI_chucun[t,:] = RI_results 
    #print(i)
    #print(RI_results)
    
    #做分类
    U1 = np.ones(n)
    result.insert(4, 'U1',U1, allow_duplicates=False)
    #做预测
    U11 = np.ones(n1)
    datapre.insert(4, 'U1',U11, allow_duplicates=False)

    # 初始化存储结果的数据结构
    ACC = {i: [] for i in range(1, 8)}
    ACCp = {i: [] for i in range(1, 8)}
    Pre = {i: [] for i in range(1, 8)}
    classifiers = {i: None for i in range(1, 8)}

    ACC_all = []
    ACCp_all = []

    # 统一配置参数
    svm_params = {
        'kernel': 'linear',
        'C': 1
    }
    grid_params = {
        'x_min': 0,
        'x_max': 1,
        'y_min': 0,
        'y_max': 1,
        'step': 0.01
    }
    thresholds = {
        1: 0,          # alpha1阈值
        2: -0.75,      # alpha2阈值
        3:  -0.75,          # 其他alpha阈值
        4:  -0.75,          # 按实际需求修改
        5: -0.75,
        6: -0.75,
        7: -0.75
    }

    # 公共特征列
    feature_cols = ['U', 'U1']

    # 主处理循环
    for alpha_num in range(1, 8):
        # 生成训练标签
        label = np.where(result[f'alpha{alpha_num}'] >= thresholds[alpha_num], 1, -1)
        #label_o[alpha_num].append[label]
        # 训练分类器
        clf = SVC(**svm_params)
        clf.fit(result[feature_cols], label)
        classifiers[alpha_num] = clf

        # 计算训练准确率
        train_pred = clf.predict(result[feature_cols])
        ACC_all.append(train_pred - label)

        ACC[alpha_num].append(1 - np.count_nonzero(train_pred - label) / len(label))

        # 生成预测标签
        pre_label = np.where(datapre[f'alpha{alpha_num}'] >= thresholds[alpha_num], 1, -1)
        #label_pre[alpha_num].append[pre_label]
        # 计算预测准确率
        test_pred = clf.predict(datapre[feature_cols])
        ACCp[alpha_num].append(1 - np.count_nonzero(test_pred - pre_label) / len(pre_label))

        ACCp_all.append(test_pred - pre_label)

        # 生成预测网格
        xx, yy = np.meshgrid(
            np.arange(grid_params['x_min'], grid_params['x_max'], grid_params['step']),
            np.arange(grid_params['y_min'], grid_params['y_max'], grid_params['step'])
        )
        grid_points = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=['U', 'U1'])  # 关键修改

        #Pre[alpha_num].append(clf.predict(np.c_[xx.ravel(), yy.ravel()]))
        Pre[alpha_num].append(clf.predict(grid_points))
    for i in range(1,8):
        globals()[f'ACC{i}_cunchu'][t] = np.array(ACC[i])
        globals()[f'ACCp{i}_cunchu'][t] = np.array(ACCp[i])
        globals()[f'Pre{i}_cunchu'][t,:] = np.array(Pre[i]).flatten()
    #训练集的总正确率
    acc1_final = np.array(np.linalg.norm(x=np.array(ACC_all).T, ord=1, axis=1, keepdims=False))
    acc_final = np.sum(acc1_final==0)/len(acc1_final)
    ACC_finalcunchu.append(acc_final)
    
    #print(acc_final)
    #测试集的总准确率
    acc1_prefinal = np.array(np.linalg.norm(x=np.array(ACCp_all).T, ord=1, axis=1, keepdims=False))
    acc_prefinal = np.sum(acc1_prefinal==0)/len(acc1_prefinal)
    ACC_prefinalcunchu.append(acc_prefinal)
    #print(acc_prefinal)
    
    # 生成 Z1-Z7 的 numpy 数组
    zs = [np.array(getattr(result, f'Z{i}')) for i in range(1, 8)]

    # 生成 I 的指标函数结果
    I_or = {}
    for i in range(1, 8):
        alpha_origin = getattr(result, f'alpha{i}_origin')
        # 根据序号选择函数
        if i == 1:
            func1, func2 = I_11or, I_12or
        else:
            func1, func2 = I_21or, I_22or
        # 生成 I1 和 I2 序列
        I1 = [func1(val) for val in alpha_origin]
        I2 = [func2(val) for val in alpha_origin]
        I_or[(i, 1)] = I1
        I_or[(i, 2)] = I2

    # 生成 zI 交叉项结果
    zI = {}
    for i in range(1, 8):
        z = zs[i-1]
        for j in (1, 2):
            key = (i, j)
            # 动态生成变量名存储到字典
            zI[f'zI_{i}{i}{j}or'] = z * I_or[key]
            # 若需要保留原全局变量名，使用下面这行
            # globals()[f'zI_{i}{i}{j}or'] = z * I_or[key]
            
    # 数据准备部分
    data_modelor = [
        result['Y'].values,
        result['X0'].values,
        result['X1'].values,
        result['X2'].values
    ]

    # 动态添加交叉项（假设已按之前方案生成zI字典）
    for i in range(1, 8):
        data_modelor.append(zI[f'zI_{i}{i}1or'])
        data_modelor.append(zI[f'zI_{i}{i}2or'])

    # 创建DataFrame
    columns = ['Y','X0','X1','X2'] 
    columns += [f'z{i}*I{i}{j}' for i in range(1,8) for j in (1,2)]
    df_model = pd.DataFrame(np.array(data_modelor).T, columns=columns)

    # 回归分析
    model = sm.OLS(df_model['Y'], df_model[df_model.columns[1:]])
    results = model.fit()

    # 参数提取
    params = results.params


    #beta_allor= {'beta0': [params.iloc[0]],  # 使用 .iloc 按位置访问'beta1': [params.iloc[1]],  # 索引从0开始 'beta2': [params.iloc[2]]   # 确保params是Series/DataFrame
   # }
    beta_allor = np.array(params.iloc[:3]) 
    betaor_chucun[t,:] = beta_allor #存储beta结果

    
    alpha_allor = {}
    for i in range(1, 8):
        for j in range(1, 3):
            # 计算绝对位置
            pos = 3 + 2*(i-1) + (j-1)
            # 使用 .iloc 按位置访问
            alpha_allor[f'alpha{i}_{j}'] = [params.iloc[pos]]
        
    
    for i in range(1, 8):
        for j in range(1, 3):
            globals()[f'alphaor{i}_{j}cunchu'].append(alpha_allor[f'alpha{i}_{j}'])
    #将结果存储
    
    # 生成 Z1-Z7 的 numpy 数组
    zs = [np.array(getattr(result, f'Z{i}')) for i in range(1, 8)]

    # 生成 I 的指标函数结果
    I = {}
    for i in range(1, 8):
        label = np.where(result[f'alpha{alpha_num}'] >= thresholds[alpha_num], 1, -1)
        # 根据序号选择函数
        if i == 1:
            func1, func2 = I_11, I_12
        else:
            func1, func2 = I_21, I_22
        # 生成 I1 和 I2 序列
        I1 = [func1(val) for val in label]
        I2 = [func2(val) for val in label]
        I[(i, 1)] = I1
        I[(i, 2)] = I2

    # 生成 zI 交叉项结果
    zI = {}
    for i in range(1, 8):
        z = zs[i-1]
        for j in (1, 2):
            key = (i, j)
            # 动态生成变量名存储到字典
            zI[f'zI_{i}{i}{j}'] = z * I[key]
            # 若需要保留原全局变量名，使用下面这行
            # globals()[f'zI_{i}{i}{j}or'] = z * I_or[key]


    # 数据准备部分
    data_model = [
        result['Y'].values,
        result['X0'].values,
        result['X1'].values,
        result['X2'].values
    ]

    # 动态添加交叉项（假设已按之前方案生成zI字典）
    for i in range(1, 8):
        data_model.append(zI[f'zI_{i}{i}1'])
        data_model.append(zI[f'zI_{i}{i}2'])

    # 创建DataFrame
    columns = ['Y','X0','X1','X2'] 
    columns += [f'z{i}*I{i}{j}' for i in range(1,8) for j in (1,2)]
    df_model = pd.DataFrame(np.array(data_model).T, columns=columns)

    # 回归分析
    model = sm.OLS(df_model['Y'], df_model[df_model.columns[1:]])
    results = model.fit()

    # 参数提取
    params = results.params  
    beta_all = np.array(params.iloc[:3]) 
    beta_chucun[t,:] = beta_allor #存储beta结果
    
    alpha_all = {}
    for i in range(1, 8):
        for j in range(1, 3):
            # 计算绝对位置
            pos = 3 + 2*(i-1) + (j-1)
            # 使用 .iloc 按位置访问
            alpha_all[f'alpha{i}_{j}'] = [params.iloc[pos]]
    for i in range(1, 8):
        for j in range(1, 3):
            globals()[f'alpha{i}_{j}cunchu'].append(alpha_all[f'alpha{i}_{j}'])
    #将结果存储 

end = time.perf_counter()
execution_time_1 = (end - start)/200
 

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199


In [3]:
# 初始化结果容器
result_need = []

# 按顺序添加数据（必须与列名顺序严格对应）
# -------------------------------------------
# 添加 K1-K7（假设 K_chucun 是包含7个元素的列表）
result_need.extend(K_chucun.T)  # 使用 extend 而不是 append (10,7)

# 添加 RI1-RI7（假设 RI_chucun 是包含7个元素的列表）
result_need.extend(RI_chucun.T) #(10,7)

# 添加 beta_or（假设 betaor_chucun 是包含3个元素的列表：[beta0_or, beta1_or, beta2_or]）
result_need.extend(betaor_chucun.T)   #(10,3)

# 添加 alpha{i}_{j}or（i=1-7, j=1-2）
for i in range(1, 8):
    for j in range(1, 3):
        result_need.append(np.array(globals()[f'alphaor{i}_{j}cunchu']).flatten())

# 添加 beta（假设 beta_chucun 是包含3个元素的列表：[beta0, beta1, beta2]）
result_need.extend(beta_chucun.T)  # 需要确保长度=3

# 添加 alpha{i}_{j}（i=1-7, j=1-2）
for i in range(1, 8):
    for j in range(1, 3):
        result_need.append(np.array(globals()[f'alpha{i}_{j}cunchu']).flatten())

# 添加 ACC1-ACC7
for i in range(1, 8):
    result_need.append(np.array(globals()[f'ACC{i}_cunchu']).flatten())

# 添加 ACCP1-ACCP7
for i in range(1, 8):
    result_need.append(np.array(globals()[f'ACCp{i}_cunchu']).flatten())

# 添加最终的 ACC 和 ACCP
result_need.append(ACC_finalcunchu)
result_need.append(ACC_prefinalcunchu)

# 构建列名（必须与上述添加顺序完全一致）
# -------------------------------------------
columns = (
    [f'K{i}' for i in range(1, 8)] +          # K1-K7
    [f'RI{i}' for i in range(1, 8)] +         # RI1-RI7
    ['beta0_or', 'beta1_or', 'beta2_or'] +    # beta_or
    [f'alpha{i}_{j}or' for i in range(1,8) for j in (1,2)] +  # alphaor
    ['beta0', 'beta1', 'beta2'] +             # beta
    [f'alpha{i}_{j}' for i in range(1,8) for j in (1,2)] +    # alpha
    [f'ACC{i}' for i in range(1,8)] +         # ACC1-ACC7
    [f'ACCP{i}' for i in range(1,8)] +        # ACCP1-ACCP7
    ['ACC', 'ACCP']                           # 最终结果
)
save_need1 = pd.DataFrame(np.array(result_need).T,columns = columns)
save_need1
save_need1.to_csv('high dimension canshu with n = 400.csv',index=False,header=True) 
data1 = save_need1

In [4]:
# 自动处理Pre1_cunchu到PreN_cunchu
num_files = 7  # 根据实际情况修改
for i in range(1, num_files + 1):
    data = globals()[f'Pre{i}_cunchu']
    pd.DataFrame(data).to_csv(f'juece{i}.csv', index=False, header=True)